# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W

# Criando o database silver

In [0]:
%sql
USE CATALOG projeto;
-- DROP DATABASE silver CASCADE; -- Executar se tiver uma silver já criada
CREATE DATABASE IF NOT EXISTS silver;

## Tabela: Atendentes

## Tabela: Canais

Para a tabela de dm.canais, é preciso garantir que os tipos de dados bem como os nomes das colunas estejam corretos conforme o grupo definiu.

Schema final da tabela:

```
root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)
```

In [ ]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, upper, initcap, coalesce

# criação da sessão do Spark
spark = SparkSession.builder.appName("bronze_to_silver").getOrCreate()

In [ ]:
# paths dos schemas
bronze_path = "workspace.bronze"
silver_path = "workspace.silver"

In [ ]:
canais = spark.read.table(f"{bronze_path}.canais")

In [ ]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

root
 |-- nome_canal: string (nullable = true)
 |-- canal_status: string (nullable = true)



In [ ]:
canais.display() # checando os dados

nome_canal,canal_status
URA,ativo
ATENDIMENTO INICIAL,ativo
ATENDIMENTO ESPECIALIZADO,invativo
CHATBOT,ativo
WEB,invativo
email,inativo


De acordo com o observado acima, a coluna ```nome_canal``` possui uma despadronização quanto à forma que as palavras estão escritas. Para padronizar, vamos capitalizar todas elas e caso tenha alguma ocorrência futura que seja nula, vamos colocar o valor de ```Desconhecido```.

Já na coluna de ```canal_status```, existe uma padronização quanto às diferentes escritas de inativo (sejam corretas no vocabulário português ou não). Para padronizar, vamos checar a primeira letra de cada ocorrência e atribuir à um valor constante, que será ```Ativo```, ```Inativo``` ou ```Desconhecido```, caso o valor da coluna seja nulo.

In [ ]:
canais = (
    canais
    # nome_canal
    .withColumn("nome_canal", 
                # se o nome do canal for nulo, substitui por desconhecido
                coalesce(initcap(col("nome_canal")), lit("Desconhecido"))
    )
    .withColumnRenamed("nome_canal", "Nome_Canal")
    .withColumn("Nome_Canal", col("Nome_Canal").cast(StringType()))
    
    # canal_status
    .withColumn("canal_status", 
                when(upper(col("canal_status")).startswith("A"), "Ativo")
                .when(upper(col("canal_status")).startswith("I"), "Inativo")
                .otherwise("Desconhecido")
    )
    .withColumnRenamed("canal_status", "Status_Canal")
)

In [ ]:
# checando alterações pós tratamento
canais.display()

Nome_Canal,Status_Canal
Ura,Ativo
Atendimento Inicial,Ativo
Atendimento Especializado,Inativo
Chatbot,Ativo
Web,Inativo
Email,Inativo


In [ ]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)



Todas as mudanças foram efetivas e deixou a coluna padronizada para o futuro.

Como existem somente 6 ocorrências dos dados, não é possível criar futuras Views somente com esta tabela, somente em conjunto de outras tabelas.

Com isso, resta partir para o salvamento da tabela na Silver Layer.

In [ ]:
canais.write.format("delta").mode("overwrite").saveAsTable(f"{silver_path}.dim_canais")

## Tabela: Chamados

## Tabela: Chamados_Hora

In [0]:
# Lendo a tabela chamados_hora da camada bronze e visualizando os primeiros registros
df_bz = spark.table("bronze.chamados_hora")
print(f"bronze.chamados_hora: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
# Remoção de duplicatas:
df_max = (
    df_bz.groupBy("ID_Chamado")
         .agg(F.max("ingestion_timestamp").alias("ingestion_timestamp"))
)
df_bz = df_bz.join(df_max, on=["ID_Chamado", "ingestion_timestamp"], how="inner")


# Renomeando as colunas para snake_case
df = (
    df_bz
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado_raw")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento_raw")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento_raw")
    # ingestion_timestamp já está em snake_case
)

# Criando uma função para tirar o " �s " e converter pra timestamp
def to_ts(col):
    return F.to_timestamp(F.regexp_replace(col, " �s ", " "), "dd/MM/yyyy HH:mm:ss")

df = (
    # Aplicando a função nas três colunas de tempo e dropando as raws
    df
    .withColumn("data_hora_abertura", to_ts(F.col("hora_abertura_chamado_raw")))
    .withColumn("data_hora_inicio_atendimento", to_ts(F.col("hora_inicio_atendimento_raw")))
    .withColumn("data_hora_finalizacao_atendimento", to_ts(F.col("hora_finalizacao_atendimento_raw")))
    .drop(
        "hora_abertura_chamado_raw",
        "hora_inicio_atendimento_raw",
        "hora_finalizacao_atendimento_raw"
    )

    # garantir tipos dos IDs em long
    .withColumn("id_chamado", F.col("id_chamado").cast("long"))
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))

    # Criando duas métricas úteis de minutos
    .withColumn(
        "tempo_espera_atendimento_min",
        F.round((F.col("data_hora_inicio_atendimento").cast("long") - F.col("data_hora_abertura").cast("long")) / 60.0, 2)
    )

    .withColumn(
        "tempo_atendimento_min",
        F.round(
            (F.col("data_hora_finalizacao_atendimento").cast("long") - F.col("data_hora_inicio_atendimento").cast("long")) / 60.0, 2)
    )
)

# Ordenando as colunas
df = df.select(
    "id_chamado",
    "id_cliente",
    "data_hora_abertura",
    "data_hora_inicio_atendimento",
    "data_hora_finalizacao_atendimento",
    "tempo_espera_atendimento_min",
    "tempo_atendimento_min",
    "ingestion_timestamp"
)

display(df.limit(10))


In [0]:
# Salvando no silver.dim_chamado_hora (formato delta por padrão)
df.write.mode("overwrite").saveAsTable("silver.dim_chamado_hora")
print(f"silver.dim_chamado_hora: {df.count()} rows")

## Tabela: Clientes

## Tabela: Custos

## Tabela: Motivos

## Tabela: Pesquisa_Satisfação